# Motor Exercise 4 — looking for a high-command plateau

Plot repeated settled wheel-speed measurements across high PWM values and look for a region where more command produces little extra speed.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the plotting route; it is not evidence about your robot and is not a result you should expect to reproduce. When you are ready, change only the settings in **Use the example or your own data** and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV, set it to `False`, and enter the filename. This is the main cell you need to edit.

Expected CSV columns: `wheel`, `direction`, `PWM`, `trial_num`, `sample_num`, and `settled_speed_cps`.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "motor_exercise04_saturation.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The example contains a deliberately visible plateau. It illustrates the analysis and is not evidence that your motor must saturate there.


In [ ]:
example_rows = []
for wheel, wheel_scale in [("left", 1.00), ("right", 0.96)]:
    for direction, direction_scale in [("forward", 1.00), ("reverse", 0.92)]:
        sign = 1 if direction == "forward" else -1
        for pwm_magnitude in range(40, 191, 10):
            typical_speed = (
                1050 * wheel_scale * direction_scale
                * (1 - np.exp(-(2 * pwm_magnitude - 45) / 95))
            )
            for trial_num in range(1, 6):
                for sample_num in range(6):
                    example_rows.append({
                        "wheel": wheel,
                        "direction": direction,
                        "PWM": sign * pwm_magnitude,
                        "trial_num": trial_num,
                        "sample_num": sample_num,
                        "settled_speed_cps": sign * (
                            typical_speed + rng.normal(0, 10)
                        ),
                    })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

This is where your uploaded CSV enters the notebook. Check the first rows before continuing: column names, units and labels should match the exercise.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Choose one wheel and direction


In [ ]:
SELECTED_WHEEL = "left"
SELECTED_DIRECTION = "forward"

selected = data.loc[
    (data["wheel"] == SELECTED_WHEEL)
    & (data["direction"] == SELECTED_DIRECTION)
].copy()


## 5. Plot the repeated settled-speed observations

The individual points show how much repeated trials vary at each command.


In [ ]:
sns.stripplot(
    data=selected,
    x="PWM",
    y="settled_speed_cps",
    jitter=0.16,
    alpha=0.45,
    native_scale=True,
)
plt.title(f"Settled motor response: {SELECTED_WHEEL}, {SELECTED_DIRECTION}")
plt.xlabel("Requested PWM")
plt.ylabel("Settled encoder speed (counts/s)")
plt.show()


## 6. Compare neighbouring command values

The summary keeps one mean and spread per PWM. `diff()` then shows how much the typical speed changed from the previous tested command.


In [ ]:
response_summary = (
    selected.groupby("PWM", as_index=False)
    .agg(
        typical_speed_cps=("settled_speed_cps", "mean"),
        repeat_spread_cps=("settled_speed_cps", "std"),
    )
    .sort_values("PWM")
)
response_summary["speed_change_cps"] = response_summary["typical_speed_cps"].diff()

response_summary


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

axes[0].errorbar(
    response_summary["PWM"],
    response_summary["typical_speed_cps"],
    yerr=response_summary["repeat_spread_cps"],
    marker="o",
    capsize=3,
)
axes[0].set(title="Mean settled speed and repeat spread", ylabel="Speed (counts/s)")

sns.lineplot(
    data=response_summary,
    x="PWM",
    y="speed_change_cps",
    marker="o",
    ax=axes[1],
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Extra speed from each neighbouring PWM step",
    xlabel="Requested PWM",
    ylabel="Speed change (counts/s)",
)
plt.tight_layout()
plt.show()


## What to notice

- Does the response remain flat across several neighbouring commands?
- Is the remaining speed change small compared with repeat-to-repeat spread?
- What safely tested interval bounds the possible plateau?
- A fastest observed point alone does not demonstrate saturation.
